# Final Completed Pipeline
Consists of:
* Retrospective Labelling 
  * Selected Best Model: LightGBM
  * Tuned on: (1) pr-auc, (2) log loss
  * Best hyperparameters: 
  * Best ensemble weights:

* Forward Classification:
  * Soft Label Treatment:
    * Soft Label Weights: 0.5
    * Soft Label Cut Off: 0.7
  * Historical recency weight: 0.03
  * Selected Models: LightGBM, XGBoost, Extra Trees
  * Submission 1: LightGBM Only
  * Submission 2: Best Ensemble

In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier

In [2]:
train_raw = pd.read_csv("train.csv")
test_raw = pd.read_csv("test.csv")
edges_raw = pd.read_csv("txs_edgelist.csv")

print("train_raw:", train_raw.shape)
print("test_raw:", test_raw.shape)
print("edges_raw:", edges_raw.shape)
print("Train columns:", train_raw.columns.tolist()[:10], "...")


train_raw: (141772, 168)
test_raw: (15329, 168)
edges_raw: (234355, 2)
Train columns: ['txId', 'time_step', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8'] ...


In [3]:
# Match the original data-preparation logic.
# Keep train_raw unchanged: these node features are used only for
# retrospective soft-label generation.
train_raw_node_info = train_raw.copy()

edges_for_nodes = edges_raw.drop_duplicates(
    subset=["txId1", "txId2"]
).copy()
edges_for_nodes = edges_for_nodes[
    edges_for_nodes["txId1"] != edges_for_nodes["txId2"]
]

in_deg = edges_for_nodes["txId2"].value_counts()
out_deg = edges_for_nodes["txId1"].value_counts()

node_cols_for_soft_labelling = [
    "in_degree",
    "out_degree",
    "total_degree",
    "degree_ratio",
    "timestep_tx_volume",
]

train_raw_node_info["in_degree"] = (
    train_raw_node_info["txId"].map(in_deg).fillna(0).astype("float32")
)
train_raw_node_info["out_degree"] = (
    train_raw_node_info["txId"].map(out_deg).fillna(0).astype("float32")
)
train_raw_node_info["total_degree"] = (
    train_raw_node_info["in_degree"] + train_raw_node_info["out_degree"]
)
train_raw_node_info["degree_ratio"] = (
    (train_raw_node_info["in_degree"] + 1.0)
    / (train_raw_node_info["out_degree"] + 1.0)
)

# Match data_prep(4): volume is calculated from train_raw only.
step_vol = train_raw_node_info["time_step"].value_counts()
train_raw_node_info["timestep_tx_volume"] = (
    train_raw["time_step"].map(step_vol).astype("float32")
)

print("Node columns added:", node_cols_for_soft_labelling)
print("train_raw_node_info shape:", train_raw_node_info.shape)
display(train_raw_node_info[["txId", "time_step"] + node_cols_for_soft_labelling].head())


Node columns added: ['in_degree', 'out_degree', 'total_degree', 'degree_ratio', 'timestep_tx_volume']
train_raw_node_info shape: (141772, 173)


,txId,time_step,in_degree,out_degree,total_degree,degree_ratio,timestep_tx_volume
0,230425980,1,1.0,1.0,2.0,1.000000,7880.0
1,5530458,1,1.0,1.0,2.0,1.000000,7880.0
2,232022460,1,1.0,2.0,3.0,0.666667,7880.0
3,232438397,1,160.0,1.0,161.0,80.500000,7880.0
4,230460314,1,2.0,8.0,10.0,0.333333,7880.0


In [4]:
# Soft-label generation
SOFT_LABEL_WEIGHT = 0.5
CONFIDENCE_CUTOFF = 0.7
RECENCY_DECAY = 0.03

# Feature settings for final prediction
USE_GRAPH_FEATURES = True
USE_TIME_STEP = False

# Submission paths
LGBM_SUBMISSION_PATH = "submission_final_lightgbm.csv"
ENSEMBLE_SUBMISSION_PATH = "submission_final_lgbm_dominant_ensemble.csv"

# Model used to generate retrospective soft labels
pr_auc_params = {
    "scale_pos_weight": 3.0,
    "num_leaves": 31,
    "n_estimators": 500,
    "min_child_samples": 50,
    "max_depth": 20,
    "learning_rate": 0.1,
}

log_loss_params = {
    "scale_pos_weight": 3.0,
    "num_leaves": 15,
    "n_estimators": 500,
    "min_child_samples": 20,
    "max_depth": 10,
    "learning_rate": 0.03,
}

weight_log_loss = 0.70
weight_pr_auc = 0.30

# Final selected LightGBM model
best_lightgbm_params = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 15,
    "max_depth": 10,
    "min_child_samples": 20,
    "min_split_gain": 0.1,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
}

# Final selected XGBoost model
xgboost_params = {
    "subsample": 1.0,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "n_estimators": 300,
    "min_child_weight": 5,
    "max_depth": 7,
    "learning_rate": 0.03,
    "gamma": 0.0,
    "colsample_bytree": 0.8,
}

# Final selected ExtraTrees model
extra_trees_params = {
    "n_estimators": 500,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "min_impurity_decrease": 0.0,
    "max_features": 0.5,
    "max_depth": 40,
    "class_weight": "balanced",
}

ensemble_weights = {
    "LightGBM": 0.5,
    "XGBoost": 0.3,
    "ExtraTrees": 0.2,
}


In [5]:
# `node_cols_for_soft_labelling` was created in the previous cell.

# Use all numeric raw features for retrospective soft-label generation.
def get_X_for_ids(df, ids):
    indexed = df.set_index("txId", drop=False)
    aligned = indexed.loc[list(ids)]
    excluded = {"txId", "index", "label"}
    feature_columns = [
        c for c in aligned.select_dtypes(include=np.number).columns
        if c not in excluded
    ]
    return aligned[feature_columns]

def create_lgbm_pipeline(params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            objective="binary",
            random_state=100,
            verbosity=-1,
            n_jobs=-1,
            **params,
        )),
    ])

known = train_raw_node_info.loc[train_raw_node_info["label"].notna()].copy()
unlabelled = train_raw_node_info.loc[train_raw_node_info["label"].isna()].copy()

known_ids = known["txId"].tolist()
unlabelled_ids = unlabelled["txId"].tolist()

X_known = get_X_for_ids(train_raw_node_info, known_ids)
y_known = known.set_index("txId").loc[known_ids, "label"].astype(int)
X_unlabelled = get_X_for_ids(train_raw_node_info, unlabelled_ids)

pr_auc_model = create_lgbm_pipeline(pr_auc_params)
log_loss_model = create_lgbm_pipeline(log_loss_params)
pr_auc_model.fit(X_known, y_known)
log_loss_model.fit(X_known, y_known)

p_pr_auc = pr_auc_model.predict_proba(X_unlabelled)[:, 1]
p_log_loss = log_loss_model.predict_proba(X_unlabelled)[:, 1]
p_illicit = (
    weight_log_loss * p_log_loss
    + weight_pr_auc * p_pr_auc
)

# Remove only legacy/static node columns from the exported soft-label file.
output_node_cols = [
    c for c in node_cols_for_soft_labelling
    if c in train_raw.columns
]
# Export the untouched raw transaction data plus soft-label columns.
train_soft_label = train_raw.copy()
train_soft_label["p_illicit"] = np.nan
train_soft_label["label_source"] = "observed"
train_soft_label.loc[unlabelled.index, "p_illicit"] = p_illicit
train_soft_label.loc[unlabelled.index, "label_source"] = "model_probability"

train_soft_label.to_csv("train_soft_label.csv", index=False)

print("Confirmed rows:", len(known))
print("Soft-labelled rows:", len(unlabelled))
print("Saved train_soft_label.csv")


Confirmed rows: 31235
Soft-labelled rows: 110537
Saved train_soft_label.csv


In [6]:
node_times = pd.concat(
    [
        train_raw[["txId", "time_step"]],
        test_raw[["txId", "time_step"]],
    ],
    ignore_index=True,
).drop_duplicates("txId").set_index("txId")["time_step"]

edges = edges_raw[["txId1", "txId2"]].copy()
edges["time_1"] = edges["txId1"].map(node_times)
edges["time_2"] = edges["txId2"].map(node_times)

graph_feature_cols = [
    "in_degree",
    "out_degree",
    "total_degree",
    "degree_ratio",
]

def temporal_node_features(cutoff_step):
    usable_edges = edges[
        edges["time_1"].notna()
        & edges["time_2"].notna()
        & (edges["time_1"] <= cutoff_step)
        & (edges["time_2"] <= cutoff_step)
    ]

    in_degree = usable_edges.groupby("txId2").size().rename("in_degree")
    out_degree = usable_edges.groupby("txId1").size().rename("out_degree")

    features = pd.concat([in_degree, out_degree], axis=1).fillna(0)
    features["total_degree"] = (
        features["in_degree"] + features["out_degree"]
    )
    features["degree_ratio"] = (
        features["out_degree"]
        / features["in_degree"].replace(0, np.nan)
    )
    features["degree_ratio"] = (
        features["degree_ratio"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return features.reset_index().rename(columns={"index": "txId"})

def add_temporal_graph_features(frame, cutoff_step):
    graph_features = temporal_node_features(cutoff_step)
    result = frame.merge(graph_features, on="txId", how="left")
    result[graph_feature_cols] = result[graph_feature_cols].fillna(0)
    return result

def calculate_recency_weights(time_steps, reference_time_step, decay_rate):
    age = reference_time_step - time_steps
    return np.exp(-decay_rate * age)


In [7]:
train_soft_label_raw = train_soft_label.copy()

observed_mask = train_soft_label_raw["label"].notna()
soft_label_mask = (
    train_soft_label_raw["label"].isna()
    & train_soft_label_raw["p_illicit"].notna()
)
confidence_mask = (
    (train_soft_label_raw["p_illicit"] >= CONFIDENCE_CUTOFF)
    | (train_soft_label_raw["p_illicit"] <= 1 - CONFIDENCE_CUTOFF)
)
confident_pseudo_mask = soft_label_mask & confidence_mask

observed_rows = train_soft_label_raw.loc[observed_mask].copy()
observed_rows["training_target"] = observed_rows["label"].astype(int)
observed_rows["label_source_weight"] = 1.0

pseudo_rows = train_soft_label_raw.loc[confident_pseudo_mask].copy()
pseudo_rows["training_target"] = (
    pseudo_rows["p_illicit"] >= 0.5
).astype(int)
pseudo_rows["label_source_weight"] = SOFT_LABEL_WEIGHT

final_train = pd.concat(
    [observed_rows, pseudo_rows],
    ignore_index=True,
)

print("Confirmed rows:", len(observed_rows))
print("Confident pseudo-labelled rows:", len(pseudo_rows))
print("Total final training rows:", len(final_train))


Confirmed rows: 31235
Confident pseudo-labelled rows: 105509
Total final training rows: 136744


In [8]:
final_train = add_temporal_graph_features(
    final_train,
    cutoff_step=int(final_train["time_step"].max()),
)

test_features = []
for step, step_rows in test_raw.groupby("time_step", sort=True):
    step_rows = add_temporal_graph_features(
        step_rows.copy(),
        cutoff_step=int(step),
    )
    test_features.append(step_rows)

test_features = pd.concat(test_features, ignore_index=True)

feature_cols = [
    c for c in train_raw.columns
    if c not in [
        "index", "txId", "label", "time_step",
        *[c for c in node_cols_for_soft_labelling if c in train_raw.columns],
    ]
]

final_feature_cols = list(feature_cols)
if USE_GRAPH_FEATURES:
    final_feature_cols += graph_feature_cols
if USE_TIME_STEP:
    final_feature_cols += ["time_step"]

X_final = final_train[final_feature_cols]
y_final = final_train["training_target"].astype(int)
X_test = test_features[final_feature_cols]

print("Feature count:", len(final_feature_cols))
print("Training shape:", X_final.shape)
print("Test shape:", X_test.shape)


Feature count: 169
Training shape: (136744, 169)
Test shape: (15329, 169)


In [9]:
class_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_final,
)

recency_weights = calculate_recency_weights(
    final_train["time_step"].to_numpy(),
    reference_time_step=int(final_train["time_step"].max()),
    decay_rate=RECENCY_DECAY,
)

source_weights = final_train["label_source_weight"].to_numpy()

final_sample_weights = (
    class_weights * recency_weights * source_weights
)
final_sample_weights = (
    final_sample_weights / final_sample_weights.mean()
)

print("Sample-weight minimum:", final_sample_weights.min())
print("Sample-weight maximum:", final_sample_weights.max())


Sample-weight minimum: 0.2389010638761486
Sample-weight maximum: 17.480467126166644


In [10]:
final_lgbm_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "model",
        LGBMClassifier(
            objective="binary",
            random_state=100,
            n_jobs=-1,
            verbosity=-1,
            **best_lightgbm_params,
        ),
    ),
])

final_lgbm_model.fit(
    X_final,
    y_final,
    model__sample_weight=final_sample_weights,
)

lgbm_predictions = final_lgbm_model.predict_proba(X_test)[:, 1]
lgbm_predictions = np.clip(lgbm_predictions, 0.0, 1.0)

lgbm_submission = pd.DataFrame({
    "index": test_raw["index"].to_numpy(),
    "target": lgbm_predictions,
})
lgbm_submission.to_csv(
    LGBM_SUBMISSION_PATH,
    index=False,
)

display(lgbm_submission.head())
print("Saved:", LGBM_SUBMISSION_PATH)
print("Unique predictions:", lgbm_submission["target"].nunique())


,index,target
0,0,0.000278
1,1,0.000723
2,2,0.000890
3,3,0.000421
4,4,0.000017


Saved: submission_final_lightgbm.csv
Unique predictions: 15281


In [11]:
models = {
    "LightGBM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            objective="binary",
            random_state=100,
            n_jobs=-1,
            verbosity=-1,
            **best_lightgbm_params,
        )),
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=100,
            n_jobs=-1,
            **xgboost_params,
        )),
    ]),
    "ExtraTrees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            random_state=100,
            n_jobs=-1,
            **extra_trees_params,
        )),
    ]),
}

model_predictions = {}
for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(
        X_final,
        y_final,
        model__sample_weight=final_sample_weights,
    )
    model_predictions[model_name] = model.predict_proba(X_test)[:, 1]
    print(f"Finished {model_name}.")

prediction_frame = pd.DataFrame(model_predictions)
for model_name in models:
    prediction_frame[f"{model_name}_rank"] = (
        prediction_frame[model_name].rank(pct=True)
    )

prediction_frame["ensemble_target"] = sum(
    ensemble_weights[model_name]
    * prediction_frame[f"{model_name}_rank"]
    for model_name in models
)

ensemble_submission = pd.DataFrame({
    "index": test_raw["index"].to_numpy(),
    "target": prediction_frame["ensemble_target"].to_numpy(),
})
ensemble_submission.to_csv(
    ENSEMBLE_SUBMISSION_PATH,
    index=False,
)

display(ensemble_submission.head())
print("Saved:", ENSEMBLE_SUBMISSION_PATH)
print("Unique predictions:", ensemble_submission["target"].nunique())


Training LightGBM...
Finished LightGBM.
Training XGBoost...
Finished XGBoost.
Training ExtraTrees...
Finished ExtraTrees.


,index,target
0,0,0.498898
1,1,0.695851
2,2,0.672562
3,3,0.566527
4,4,0.175563


Saved: submission_final_lgbm_dominant_ensemble.csv
Unique predictions: 14824


In [12]:
from pathlib import Path


def compare_csv_outputs(generated_path, reference_path, key_columns):
    generated_path = Path(generated_path)
    reference_path = Path(reference_path)

    if not reference_path.exists():
        print(f"SKIPPED: {reference_path} was not found")
        return False

    generated = pd.read_csv(generated_path)
    reference = pd.read_csv(reference_path)

    same_columns = list(generated.columns) == list(reference.columns)
    same_shape = generated.shape == reference.shape

    if same_shape and same_columns:
        generated_sorted = generated.sort_values(key_columns).reset_index(drop=True)
        reference_sorted = reference.sort_values(key_columns).reset_index(drop=True)

        numeric_columns = generated_sorted.select_dtypes(include=np.number).columns
        non_numeric_columns = [
            c for c in generated_sorted.columns
            if c not in numeric_columns
        ]

        values_match = True
        if len(numeric_columns) > 0:
            values_match = np.allclose(
                generated_sorted[numeric_columns].to_numpy(),
                reference_sorted[numeric_columns].to_numpy(),
                rtol=1e-10,
                atol=1e-12,
                equal_nan=True,
            )
        if values_match and non_numeric_columns:
            values_match = generated_sorted[non_numeric_columns].equals(
                reference_sorted[non_numeric_columns]
            )
    else:
        values_match = False

    status = "PASS" if same_shape and same_columns and values_match else "FAIL"
    print(f"{status}: {generated_path} vs {reference_path}")
    print("  Generated shape:", generated.shape)
    print("  Reference shape:", reference.shape)
    print("  Columns match:", same_columns)
    print("  Values match:", values_match)

    if status == "FAIL" and same_shape and same_columns:
        generated_sorted = generated.sort_values(key_columns).reset_index(drop=True)
        reference_sorted = reference.sort_values(key_columns).reset_index(drop=True)
        for column in generated_sorted.columns:
            if column in key_columns:
                continue
            if pd.api.types.is_numeric_dtype(generated_sorted[column]):
                difference = np.abs(
                    generated_sorted[column] - reference_sorted[column]
                )
                print(
                    f"  {column}: max absolute difference = {difference.max():.6g}, "
                    f"different rows = {(difference > 1e-12).sum():,}"
                )
            else:
                print(
                    f"  {column}: different rows = "
                    f"{(generated_sorted[column] != reference_sorted[column]).sum():,}"
                )

    return status == "PASS"


soft_label_matches = compare_csv_outputs(
    "train_soft_label.csv",
    "train_soft_label_reference.csv",
    ["txId"],
)

lgbm_matches = compare_csv_outputs(
    LGBM_SUBMISSION_PATH,
    "attempt5_tuned_LGBM_Classify.csv",
    ["index"],
)

ensemble_matches = compare_csv_outputs(
    ENSEMBLE_SUBMISSION_PATH,
    "attempt6_tuned_ensemble.csv",
    ["index"],
)


PASS: train_soft_label.csv vs train_soft_label_reference.csv
  Generated shape: (141772, 170)
  Reference shape: (141772, 170)
  Columns match: True
  Values match: True
PASS: submission_final_lightgbm.csv vs attempt5_tuned_LGBM_Classify.csv
  Generated shape: (15329, 2)
  Reference shape: (15329, 2)
  Columns match: True
  Values match: True
PASS: submission_final_lgbm_dominant_ensemble.csv vs attempt6_tuned_ensemble.csv
  Generated shape: (15329, 2)
  Reference shape: (15329, 2)
  Columns match: True
  Values match: True


In [13]:
print("Completed successfully.")
print("LightGBM submission:", LGBM_SUBMISSION_PATH)
print("Ensemble submission:", ENSEMBLE_SUBMISSION_PATH)


Completed successfully.
LightGBM submission: submission_final_lightgbm.csv
Ensemble submission: submission_final_lgbm_dominant_ensemble.csv
